In [1]:
#!/usr/bin/env python3
"""
COMMENT 8 - survivorship breakdown of recordings dropped by the
<100-timeframe filter.

"""
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT      = Path("/data0/b2ai-voice/3.0.0")
SPEC      = ROOT / "features" / "torchaudio_mel_spectrogram.parquet"
PD_PHEN   = ROOT / "phenotype" / "diagnosis" / "parkinsons_disease.tsv"
CTRL_PHEN = ROOT / "phenotype" / "diagnosis" / "control.tsv"
DEMO_PATH = ROOT / "phenotype" / "demographics" / "demographics.tsv"

SELECTED_TASKS = ['prolonged-vowel', 'glides-high-to-low', 'glides-low-to-high',
                  'diadochokinesis-pataka', 'rainbow-passage', 'picture-description',
                  'story-recall', 'maximum-phonation-time-1']
MIN_TIME_FRAMES = 100

# ---- read only the 3 light columns (no spectrogram arrays) ----
pf = pq.ParquetFile(SPEC)
parts = [pf.read_row_group(i, columns=['participant_id', 'task_name', 'n_frames']).to_pandas()
         for i in range(pf.num_row_groups)]
spec = pd.concat(parts, ignore_index=True)
spec['participant_id'] = spec['participant_id'].astype(str).str.zfill(6)

# ---- labels (same cohort definition as the notebook) ----
pd_ids   = set(pd.read_csv(PD_PHEN,   sep="\t")['participant_id'].astype(str).str.zfill(6))
ctrl_ids = set(pd.read_csv(CTRL_PHEN, sep="\t")['participant_id'].astype(str).str.zfill(6)) - pd_ids
def lab(pid):
    return 1 if pid in pd_ids else (0 if pid in ctrl_ids else np.nan)
spec['label'] = spec['participant_id'].map(lab)

demo = pd.read_csv(DEMO_PATH, sep="\t")
demo['participant_id'] = demo['participant_id'].astype(str).str.zfill(6)
demo['age'] = pd.to_numeric(demo['age'].replace({'90 and above': 90}), errors='coerce')
demo = demo.sort_values('age', na_position='last').groupby('participant_id').first().reset_index()
spec = spec.merge(demo[['participant_id', 'age']], on='participant_id', how='left')

# ---- restrict to selected-task recordings belonging to the PD cohort ----
coh = spec[spec.task_name.isin(SELECTED_TASKS) & spec.label.isin([0, 1])].copy()
excl = coh[coh.n_frames < MIN_TIME_FRAMES]
kept = coh[coh.n_frames >= MIN_TIME_FRAMES]

print(f"Selected-task recordings: kept={len(kept)}  excluded(<100)={len(excl)} "
      f"({100*len(excl)/max(len(coh),1):.1f}%)")
print(f"Participants entirely removed by the filter: "
      f"{kept['participant_id'].nunique() < coh['participant_id'].nunique()} "
      f"(kept={kept['participant_id'].nunique()} of {coh['participant_id'].nunique()})")

print("\n-- excluded recordings by task --")
print(excl.groupby('task_name').size().rename('n_excluded').to_string())

print("\n-- excluded recordings by case/control --")
g = excl.groupby('label').agg(n_excluded=('n_frames', 'size'),
                              mean_age=('age', 'mean')).rename(index={0: 'control', 1: 'PD'})
print(g.to_string())

excl.groupby(['task_name', 'label']).size().unstack(fill_value=0)\
    .to_csv('excluded_recordings_by_task_group.csv')
print("\nsaved: excluded_recordings_by_task_group.csv")

Selected-task recordings: kept=2131  excluded(<100)=8 (0.4%)
Participants entirely removed by the filter: False (kept=253 of 253)

-- excluded recordings by task --
task_name
diadochokinesis-pataka    1
glides-high-to-low        1
glides-low-to-high        1
story-recall              5

-- excluded recordings by case/control --
         n_excluded   mean_age
label                         
control           3  53.666667
PD                5  69.400000

saved: excluded_recordings_by_task_group.csv
